# Explainable ASD Screening via Cross-Modal Feature Encoding (CNN + Grad-CAM)

This notebook demonstrates exploratory data analysis (EDA) and walks through the end-to-end pipeline:
1. **Dataset Inspection & Cleaning** (UCI Autism Screening Adult Dataset)
2. **Cross-Modal Feature Encoding** (Outer product, Difference matrix, Correlation matrix -> 224x224 RGB images)
3. **CNN Architecture & Model Training**
4. **Grad-CAM Interpretability & Feature Mapping**
5. **Held-Out Test Set Evaluation**

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

# Load cleaned data
cleaned_df = pd.read_csv('../data/processed/cleaned_data.csv')
print("Shape:", cleaned_df.shape)
cleaned_df.head()

## 1. Class Distribution and Feature Correlations

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=cleaned_df, x='Class/ASD', palette='Set2')
plt.title('ASD Screening Risk Distribution (0: Low Risk, 1: High Risk)')
plt.xlabel('Class/ASD')
plt.ylabel('Count')
plt.show()

plt.figure(figsize=(12, 10))
sns.heatmap(cleaned_df.corr(), cmap='coolwarm', annot=True, fmt='.2f', linewidths=0.5)
plt.title('Feature Correlation Matrix')
plt.show()

## 2. Visualizing Cross-Modal Encoded Images
Each patient is encoded as a 224x224 RGB image:
- **Red Channel**: Pairwise outer product interaction matrix $M_{ij} = f_i \cdot f_j$
- **Green Channel**: Absolute difference contrast matrix $M_{ij} = |f_i - f_j|$
- **Blue Channel**: Global training correlation scaled with interaction: $\frac{C_{ij} \cdot M_{ij} + 1}{2}$

In [ ]:
manifest_df = pd.read_csv('../data/processed/encoded_images_manifest.csv')
sample_c0 = manifest_df[manifest_df['class'] == 0].iloc[0]['file_path'].replace('data/', '../data/')
sample_c1 = manifest_df[manifest_df['class'] == 1].iloc[0]['file_path'].replace('data/', '../data/')

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(Image.open(sample_c0))
axes[0].set_title('Low Risk Patient (Class 0)')
axes[0].axis('off')

axes[1].imshow(Image.open(sample_c1))
axes[1].set_title('High Risk Patient (Class 1)')
axes[1].axis('off')
plt.show()

## 3. Grad-CAM Interpretability
Grad-CAM visualizes which feature interactions (represented by matrix coordinates) drove the CNN's diagnosis.

In [ ]:
gallery_img = Image.open('../results/gradcam_heatmaps/gradcam_summary_gallery.png')
plt.figure(figsize=(16, 8))
plt.imshow(gallery_img)
plt.axis('off')
plt.title('Grad-CAM Summary Gallery')
plt.show()